# Compact Intelligent Emergency Dispatch

Notebook that builds the city graph, creates ambulances, assigns ambulances to emergencies (time-aware), reconstructs paths, and writes the 10-column CSV required by the team.

In [1]:
import pandas as pd
import heapq
from collections import defaultdict
from datetime import datetime, timedelta
import os

print('Notebook ready')

Notebook ready


In [2]:
# Compact classes (from ambulance.py)
class CityGraph:
    def __init__(self):
        self.adj = defaultdict(list)
    def add_node(self, u):
        _ = self.adj[u]
    def add_edge(self, u, v, w, edge_id=None):
        self.adj[u].append((v, float(w), edge_id))
        self.adj[v].append((u, float(w), edge_id))
    def neighbors(self, u):
        return self.adj.get(u, [])

class Ambulance:
    def __init__(self, aid, location, avg_speed_kmph=40.0, capacity=1, base_station=None, status='Available'):
        self.aid = int(aid)
        self.location = int(location)
        self.avg_speed_kmph = float(avg_speed_kmph) if avg_speed_kmph is not None else 40.0
        self.capacity = int(capacity)
        self.base_station = base_station
        self.status = status
        self.available_until = None
    def is_available_at(self, when):
        if str(self.status).lower() in ('unavailable', 'out_of_service'):
            return False
        return (self.available_until is None) or (when >= self.available_until)
    def __repr__(self):
        return f"Ambulance({self.aid}@{self.location}, speed={self.avg_speed_kmph}, until={self.available_until})"

class DispatchSystem:
    def __init__(self, graph, ambulances):
        self.graph = graph
        self.ambulances = ambulances
    def _dijkstra_all_from(self, src):
        pq = [(0.0, src)]
        dist = {src: 0.0}
        prev = {src: None}
        while pq:
            d, u = heapq.heappop(pq)
            if d != dist.get(u, float('inf')):
                continue
            for v, w, _eid in self.graph.neighbors(u):
                nd = d + w
                if nd < dist.get(v, float('inf')):
                    dist[v] = nd
                    prev[v] = u
                    heapq.heappush(pq, (nd, v))
        return dist, prev
    def compute_travel_time_min(self, weight, amb, treat_weight_as='distance_km'):
        if treat_weight_as == 'travel_time_min':
            return float(weight)
        speed = amb.avg_speed_kmph if amb.avg_speed_kmph > 0 else 40.0
        return (float(weight) / speed) * 60.0
    def assign_ambulance(self, incident_loc, urgency=1, current_time=None, service_time_min=0.0, treat_weight_as='distance_km'):
        if current_time is None:
            current_time = datetime.now()
        dist_from_incident, prev = self._dijkstra_all_from(incident_loc)
        best = None
        for amb in self.ambulances:
            if not amb.is_available_at(current_time):
                continue
            d = dist_from_incident.get(amb.location, float('inf'))
            if d == float('inf'):
                continue
            key = (-int(urgency), float(d), int(amb.aid))
            if (best is None) or (key < best[0]):
                best = (key, amb, d)
        if best is None:
            return None
        _, amb, d_weight = best
        start_node = int(amb.location)
        travel_min = self.compute_travel_time_min(d_weight, amb, treat_weight_as=treat_weight_as)
        amb.available_until = current_time + timedelta(minutes=(travel_min + float(service_time_min)))
        amb.location = int(incident_loc)
        amb.status = 'OnDuty'
        path_nodes = []
        node = start_node
        while True:
            path_nodes.append(node)
            if node == incident_loc:
                break
            node = prev.get(node)
            if node is None:
                path_nodes = None
                break
        path_len = len(path_nodes) if path_nodes is not None else None
        return amb, d_weight, travel_min, start_node, path_len, path_nodes
    def reset_ambulances(self, make_available=True):
        for amb in self.ambulances:
            if make_available:
                amb.status = 'Available'
                amb.available_until = None


In [3]:
# Load datasets (edit BASE path if needed)
BASE = './Datasets'
NODES_CSV = os.path.join(BASE, 'nodes.csv')
EDGES_CSV = os.path.join(BASE, 'edges.csv')
AMBULANCES_CSV = os.path.join(BASE, 'ambulances.csv')
EMERGENCIES_CSV = os.path.join(BASE, 'emergencies.csv')

nodes_df = pd.read_csv(NODES_CSV)
edges_df = pd.read_csv(EDGES_CSV)
ambulances_df = pd.read_csv(AMBULANCES_CSV)
emergencies_df = pd.read_csv(EMERGENCIES_CSV)

print('Loaded:', len(nodes_df), 'nodes,', len(edges_df), 'edges,', len(ambulances_df), 'ambulances,', len(emergencies_df), 'emergencies')

Loaded: 120 nodes, 478 edges, 15 ambulances, 500 emergencies


In [4]:
# Flexible column detection
def get_col(df, names, default=None):
    for n in names:
        if n in df.columns:
            return n
    return default

time_col = get_col(emergencies_df, ['timestamp','time','call_time','datetime'])
service_col = get_col(emergencies_df, ['service_time_min','service_time','service_minutes'])
if time_col:
    emergencies_df[time_col] = pd.to_datetime(emergencies_df[time_col], errors='coerce')
    emergencies_df = emergencies_df.sort_values(by=time_col).reset_index(drop=True)

urgency_map = {'Critical':4, 'High':3, 'Moderate':2, 'Low':1}
print('Preprocessing done. time_col=', time_col, 'service_col=', service_col)

Preprocessing done. time_col= timestamp service_col= None


C:\Users\user\AppData\Local\Temp\ipykernel_18460\810714932.py:11: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  emergencies_df[time_col] = pd.to_datetime(emergencies_df[time_col], errors='coerce')


In [5]:
# Build graph
G = CityGraph()
for _, r in nodes_df.iterrows():
    nid = int(r.get('node_id', r.get('id')))
    G.add_node(nid)

for _, r in edges_df.iterrows():
    u = int(r.get('source_node', r.get('source', r.get('u'))))
    v = int(r.get('destination_node', r.get('destination', r.get('v'))))
    if 'distance_km' in r.index and pd.notna(r['distance_km']):
        w = r['distance_km']
    elif 'travel_time_min' in r.index and pd.notna(r['travel_time_min']):
        w = r['travel_time_min']
    else:
        w = r.get('weight', 1.0)
    edge_id = int(r['edge_id']) if 'edge_id' in r.index and pd.notna(r['edge_id']) else None
    G.add_edge(u, v, w, edge_id=edge_id)
print('Graph built. Nodes in graph:', len(G.adj))

Graph built. Nodes in graph: 120


In [6]:
# Create ambulances
BASE_SPEED = 40.0
ambs = []
for _, r in ambulances_df.iterrows():
    aid = int(r.get('ambulance_id', r.get('id')))
    loc = int(r.get('current_node', r.get('current_location', r.get('node'))))
    if 'speed_factor' in r.index and pd.notna(r['speed_factor']):
        sp = float(r['speed_factor']) * BASE_SPEED
    elif 'avg_speed_kmph' in r.index and pd.notna(r['avg_speed_kmph']):
        sp = float(r['avg_speed_kmph'])
    else:
        sp = BASE_SPEED
    ambs.append(Ambulance(aid, loc, avg_speed_kmph=sp, capacity=int(r.get('capacity', 1)), status=r.get('status', 'Available')))

ds = DispatchSystem(G, ambs)
print('Created', len(ambs), 'ambulances')

Created 15 ambulances


In [7]:
# Assignment loop producing 10-column output
results = []
for _, row in emergencies_df.iterrows():
    incident = int(row.get('location_node', row.get('location', row.get('node'))))
    urgency_str = str(row.get('urgency_level', 'Low')).strip()
    urg = urgency_map.get(urgency_str, 1)
    now = row[time_col] if time_col else datetime.now()
    service_min = 0.0
    if service_col and pd.notna(row.get(service_col)):
        try:
            service_min = float(row.get(service_col))
        except Exception:
            service_min = 0.0
    assigned = ds.assign_ambulance(incident, urg, current_time=now, service_time_min=service_min)
    if assigned is None:
        results.append((
            row.get('emergency_id', None),
            urgency_str,
            None,
            None,
            incident,
            None,
            None,
            'Unassigned',
            None,
            None
        ))
        continue
    amb, dist_w, travel_min, amb_start, path_len, path_nodes = assigned
    free_at = amb.available_until.strftime('%Y-%m-%d %H:%M:%S') if amb.available_until else None
    path_str = ','.join(map(str, path_nodes)) if path_nodes is not None else None
    results.append((
        row.get('emergency_id', None),
        urgency_str,
        amb.aid,
        free_at,
        incident,
        amb_start,
        travel_min,
        'Assigned',
        path_len,
        path_str
    ))

cols = [
    'emergency_id','urgency_level','assigned_ambulance','ambulance_free_at_what_time',
    'location_node','ambulance_start_node','eta','status','path_length_nodes','path'
]
out_df = pd.DataFrame(results, columns=cols)
out_df.head()

,emergency_id,urgency_level,assigned_ambulance,ambulance_free_at_what_time,location_node,ambulance_start_node,eta,status,path_length_nodes,path
0,454,High,5,2025-01-01 01:15:34,61,8,4.581818,Assigned,3,"8,110,61"
1,443,Critical,3,2025-01-01 02:51:59,93,66,9.995192,Assigned,4,"66,24,10,93"
2,406,High,14,2025-01-01 04:52:45,26,7,1.766355,Assigned,2,"7,26"
3,259,High,13,2025-01-01 05:45:04,52,75,6.075758,Assigned,4,"75,34,42,52"
4,334,Low,15,2025-01-01 07:01:15,17,71,3.257143,Assigned,4,"71,1,58,17"


In [8]:
# Save CSV
out_file = f"assignments_output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
out_df.to_csv(out_file, index=False)
print('Saved', out_file)
out_df.shape


Saved assignments_output_20251129_133838.csv


(500, 10)

In [10]:
def print_path(assignment_index, df_results):
    row = df_results.iloc[assignment_index]
    
    print(f"--- Emergency #{row['emergency_id']} Analysis ---")
    print(f"Urgency: {row['urgency_level']}")
    print(f"Status:  {row['status']}")
    
    if row['status'] == 'Assigned' and row['path']:
        nodes = row['path'].split(',')
        
        # DSA Style: 8 -> 110 -> 61
        arrow_path = " -> ".join(nodes)
        
        print(f"Route:   {arrow_path}")
        print(f"Steps:   {len(nodes)} nodes visited")
        print(f"Time:    {row['eta']:.2f} minutes")
    else:
        print("No path assigned.")
    print("-" * 40)

# Test it on the first few results
for i in range(5):
    print_path(i, out_df)

--- Emergency #454 Analysis ---
Urgency: High
Status:  Assigned
Route:   8 -> 110 -> 61
Steps:   3 nodes visited
Time:    4.58 minutes
----------------------------------------
--- Emergency #443 Analysis ---
Urgency: Critical
Status:  Assigned
Route:   66 -> 24 -> 10 -> 93
Steps:   4 nodes visited
Time:    10.00 minutes
----------------------------------------
--- Emergency #406 Analysis ---
Urgency: High
Status:  Assigned
Route:   7 -> 26
Steps:   2 nodes visited
Time:    1.77 minutes
----------------------------------------
--- Emergency #259 Analysis ---
Urgency: High
Status:  Assigned
Route:   75 -> 34 -> 42 -> 52
Steps:   4 nodes visited
Time:    6.08 minutes
----------------------------------------
--- Emergency #334 Analysis ---
Urgency: Low
Status:  Assigned
Route:   71 -> 1 -> 58 -> 17
Steps:   4 nodes visited
Time:    3.26 minutes
----------------------------------------
